In [20]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random
import os

os.environ["LANGCHAIN_PROJECT"] = "LangGraph Tool Callings"

load_dotenv()


True

In [21]:
llm = ChatGroq(model="llama-3.1-8b-instant")

Tools Creation

In [22]:
search_tool = DuckDuckGoSearchRun(region="us-en")

In [23]:
@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}
        
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    
    except Exception as e:
        return {"error": str(e)}

In [24]:
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA')
    Using Alpha Vantage with API key in the url.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTES&symbol={symbol}&apikye=M3LFPLT9CIPH91VR"
    
    r = requests.get(url)
    
    return r.json()

In [25]:
tools = [search_tool, calculator, get_stock_price]

llm_with_tools = llm.bind_tools(tools)


In [26]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [27]:
def chat_node(state: ChatState) -> ChatState:
    """
    LLM node that may answer or request a tool call. 
    """
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

In [28]:
tool_node = ToolNode(tools)

In [29]:
graph = StateGraph(ChatState)

In [30]:
graph.add_node("chat_node", chat_node)
graph.add_node("tool_node", tool_node)

In [31]:
graph.add_edge(START, "chat_node")

graph.add_conditional_edges("chat_node", tools_condition)

In [32]:
chatbot = graph.compile()

ValueError: At 'chat_node' node, 'tools_condition' branch found unknown target 'tools'